El Concepto Humano: La extracción es un paso más profundo. Imagina que te entregan un testamento de 5 páginas y te piden: "Subraya el nombre de todas las personas mencionadas y anota su edad en una tabla de Excel". Si el texto menciona a 3 personas, tu Excel tendrá 3 filas. Si no menciona a nadie, tu Excel estará vacío.

In [7]:
import os
from dotenv import load_dotenv
from typing import List, Optional

os y dotenv: Estas librerías trabajan juntas como un llavero. Te permiten guardar tu clave de la API (GOOGLE_API_KEY) en un archivo oculto llamado .env y traerla al código de forma segura. Es la mejor práctica para no publicar tus contraseñas accidentalmente.
typing (List, Optional): Son las reglas gramaticales de Python para los datos. Le dicen al código "esto será una lista de cosas" o "este dato es opcional, puede que exista o puede que no".

In [8]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

pydantic (BaseModel, Field): Es la fábrica de moldes. Pydantic nos permite crear reglas estrictas sobre cómo deben verse nuestros datos. Si decimos que un campo es un número, Pydantic no dejará pasar texto.
langchain_google_genai (ChatGoogleGenerativeAI): Es el puente de comunicación directo entre LangChain y los servidores de Gemini.

In [9]:
# ==========================================
# 1. PREPARACIÓN Y CONFIGURACIÓN
# ==========================================
load_dotenv()

True

In [10]:
# Inicializamos a Gemini frío (temperatura 0) para que no alucine datos
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0 
)

Aquí estamos inicializando nuestro cerebro de Inteligencia Artificial (llm).

model="gemini-2.0-flash-lite": Seleccionamos la versión "flash" de Gemini porque es increíblemente rápida y precisa para tareas de lectura.

La clave del éxito (temperature=0): La temperatura controla la creatividad. Cuando extraemos datos para una clínica, NO queremos que la IA invente información (alucinaciones). Al ponerla en 0, le decimos a Gemini: "Sé un robot calculador y cíñete estrictamente a lo que lees".

In [11]:
# ==========================================
# 2. DEFINICIÓN DE ESTRUCTURAS (MOLDES)
# ==========================================
# Definimos la estructura de UNA sola entidad (una fila en nuestra base de datos)
class Persona(BaseModel):
    """Información específica de una persona mencionada en la reserva."""
    nombre: str = Field(description="El nombre completo o nombre de pila de la persona")
    edad: Optional[int] = Field(description="La edad de la persona en números. Vacío si no se menciona en absoluto.")
    rol: Optional[str] = Field(description="El rol de la persona: 'Paciente', 'Especialista', o 'Acompañante'. Vacío si no se sabe.")

    

Creamos una clase llamada Persona. Esto representa una sola fila en tu futura base de datos o Excel.

nombre: str: El nombre tiene que ser texto (str).

edad: Optional[int]: La edad tiene que ser un número entero (int), pero al envolverla en Optional, le enseñamos a la IA a no entrar en pánico si no encuentra la edad. Simplemente la dejará en blanco (None).

El Field(description="..."): Esto es vital. Es el manual de instrucciones para la IA. Gemini leerá esta descripción para saber cómo llenar el molde. Por ejemplo, en "rol", le estamos restringiendo a solo tres opciones ('Paciente', 'Especialista', 'Acompañante').

In [12]:
# Definimos el contenedor principal (La tabla completa)
class ExtraccionPersonas(BaseModel):
    """Lista completa de personas encontradas en el mensaje o texto."""
    personas: List[Persona] = Field(description="Una lista con toda la información de las personas extraídas.")

Esta es la "caja" principal que contendrá todos nuestros moldes Persona.

List[Persona]: Le estamos diciendo a la IA: "No busques a una sola persona, busca a todas las que encuentres y devuélveme una lista de ellas".

In [13]:
# ==========================================
# 3. CREACIÓN DEL EXTRACTOR
# ==========================================
# Le colocamos las "anteojeras" a Gemini para que solo responda con esta estructura
extractor = llm.with_structured_output(ExtraccionPersonas)

Esta es la función estrella de LangChain moderno. Tomamos nuestro modelo (llm) y le aplicamos with_structured_output.

Analogía: Es como ponerle anteojeras a un caballo de carreras. A partir de ahora, el objeto extractor ya no sabe cómo conversar, ni saludar, ni hacer poemas. Su única misión en la vida es recibir texto y devolver el molde ExtraccionPersonas lleno.

In [14]:
# ==========================================
# 4. EJECUCIÓN Y PRUEBA
# ==========================================
# Un mensaje típico que podrías recibir en un sistema de reservas:
texto_prueba = """
Hola, quiero confirmar la reserva en la sede de Arequipa. 
El paciente es Carlos Mendoza, tiene 45 años y va para su sesión de fisioterapia. 
Irá acompañado de su hija Lucía. Por favor, avísenle a la nutricionista Valeria Gómez 
que él también quiere agendar con ella a las 4 PM.
"""

print("🤖 Leyendo el mensaje y extrayendo entidades...")

🤖 Leyendo el mensaje y extrayendo entidades...


Le pasamos un mensaje real, caótico y lleno de datos mezclados a nuestro extractor mediante el método .invoke().

Gemini lee el texto, revisa las reglas de Pydantic, y hace el trabajo de separar la información útil del "ruido".

In [15]:
# Ejecutamos el modelo
resultado_extraccion = extractor.invoke(texto_prueba)

# ==========================================
# 5. MOSTRAR RESULTADOS ORDENADOS
# ==========================================
print("\n✅ Extracción Completada. Datos listos para la base de datos:\n")


✅ Extracción Completada. Datos listos para la base de datos:



In [16]:
# Iteramos sobre la lista estructurada que nos devolvió Gemini
for persona in resultado_extraccion.personas:
    edad_texto = persona.edad if persona.edad is not None else "No especificada"
    rol_texto = persona.rol if persona.rol is not None else "No especificado"
    
    print(f"👤 Nombre: {persona.nombre}")
    print(f"   ├─ Edad: {edad_texto}")
    print(f"   └─ Rol:  {rol_texto}\n")

👤 Nombre: Carlos Mendoza
   ├─ Edad: 45
   └─ Rol:  Paciente

👤 Nombre: Lucía
   ├─ Edad: No especificada
   └─ Rol:  Acompañante

👤 Nombre: Valeria Gómez
   ├─ Edad: No especificada
   └─ Rol:  Especialista



Recorremos la lista de personas que nos entregó Gemini (resultado_extraccion.personas).

Manejo de nulos (if persona.edad is not None...): Como le dijimos a la IA que la edad y el rol eran opcionales (Optional), si no los encuentra, Python devolverá el valor None (Nada). Estas líneas sirven para que, al imprimir, en lugar de mostrar "None", muestre un texto amigable como "No especificada".

Finalmente, imprimimos los datos. Notarás que la IA fue lo suficientemente inteligente como para darse cuenta de que "Lucía" era la "Acompañante" y "Valeria Gómez" la "Especialista", aunque esas palabras no aparecieran literalmente junto a sus nombres en el texto original.